# MiFID Staging - Run All Modules

This notebook orchestrates all staging table creation in the correct dependency order.

**Execution Order:**
1. Module 4A: Price/Currency/Split Staging (foundation)
2. Module 4B: Non-Price Pre_Regulation_Ext Staging (reference tables)
3. Module 6: Hedge Liquidity Mapping
4. Module 5: Regulation Movements (depends on 4A split-price)

**Target schema:** `main.regtech_ops_stg` (all tables prefixed `bi_output_regtechops_`)

**Post-run:** Check validation cell for row counts across all modules.

In [0]:
dbutils.widgets.text("report_date", "2026-06-08", "Report Date (YYYY-MM-DD)")
report_date = dbutils.widgets.get("report_date")
print(f"MiFID Staging Orchestrator - report_date = {report_date}")

In [0]:
print("=" * 60)
print("STEP 1: Module 4A - Price/Currency/Split Staging")
print("=" * 60)
dbutils.notebook.run(
    "/Users/valentinosko@etoro.com/mifid-databricks-migration/notebooks/01_price_currency_staging",
    timeout_seconds=600,
    arguments={"report_date": report_date}
)
print("Module 4A complete.")

In [0]:
print("=" * 60)
print("STEP 2: Module 4B - Non-Price Pre_Regulation_Ext Staging")
print("=" * 60)
dbutils.notebook.run(
    "/Users/valentinosko@etoro.com/mifid-databricks-migration/notebooks/02_non_price_staging",
    timeout_seconds=600,
    arguments={"report_date": report_date}
)
print("Module 4B complete.")

In [0]:
print("=" * 60)
print("STEP 3: Module 6 - Hedge Liquidity Mapping")
print("=" * 60)
dbutils.notebook.run(
    "/Users/valentinosko@etoro.com/mifid-databricks-migration/notebooks/03_hedge_liquidity_staging",
    timeout_seconds=600,
    arguments={"report_date": report_date}
)
print("Module 6 complete.")

In [0]:
print("=" * 60)
print("STEP 4: Module 5 - Regulation Movements")
print("=" * 60)
dbutils.notebook.run(
    "/Users/valentinosko@etoro.com/mifid-databricks-migration/notebooks/04_regulation_movements_staging",
    timeout_seconds=600,
    arguments={"report_date": report_date}
)
print("Module 5 complete.")

In [0]:
%sql
-- Final validation: row counts across all modules
SELECT 'Module 4A' AS module, 'reg_currencyprice_ext' AS table_name, COUNT(*) AS row_count FROM main.regtech_ops_stg.bi_output_regtechops_reg_currencyprice_ext
UNION ALL SELECT 'Module 4A', 'reg_ext_dailymaxprices', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dailymaxprices
UNION ALL SELECT 'Module 4A', 'reg_ext_currencypricemaxdatewithsplit', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_currencypricemaxdatewithsplit
UNION ALL SELECT 'Module 4A', 'reg_ext_historysplitratio', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_historysplitratio
UNION ALL SELECT 'Module 4B', 'reg_ext_trade_getinstrument', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_getinstrument
UNION ALL SELECT 'Module 4B', 'reg_ext_trade_instrumentmetadata', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_trade_instrumentmetadata
UNION ALL SELECT 'Module 4B', 'reg_ext_dictionarycurrency', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrency
UNION ALL SELECT 'Module 4B', 'reg_ext_dictionarycurrencytype', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_dictionarycurrencytype
UNION ALL SELECT 'Module 4B', 'reg_ext_hedgeexecutionlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgeexecutionlog
UNION ALL SELECT 'Module 4B', 'reg_ext_hedgehbcexecutionlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcexecutionlog
UNION ALL SELECT 'Module 4B', 'reg_ext_hedgehbcorderlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_hedgehbcorderlog
UNION ALL SELECT 'Module 4B', 'reg_instruments_ext', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_instruments_ext
UNION ALL SELECT 'Module 6', 'reg_hedgeservertoliquidityaccount_ext', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_hedgeservertoliquidityaccount_ext
UNION ALL SELECT 'Module 6', 'reg_liquidtyacount_ext', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_ext
UNION ALL SELECT 'Module 6', 'reg_ext_liquidityaccountid', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityaccountid
UNION ALL SELECT 'Module 6', 'reg_ext_liquidityproviders', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_ext_liquidityproviders
UNION ALL SELECT 'Module 5', 'reg_migrationinout_population', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population
UNION ALL SELECT 'Module 5', 'reg_regulationinoutdailydata', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_regulationinoutdailydata
UNION ALL SELECT 'Module 5', 'reg_regulation_movments_positions', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_regulation_movments_positions
ORDER BY module, table_name